In [2]:
import pandas as pd
import numpy as np
import os

# Read splits
train_df = pd.read_csv(os.path.join("artifacts", "train.csv"))
val_df = pd.read_csv(os.path.join("artifacts", "val.csv"))
test_df = pd.read_csv(os.path.join("artifacts", "test.csv"))

print(f"Loaded Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

Loaded Train: (67534, 20), Val: (14472, 20), Test: (14472, 20)


In [3]:
def extract_features(df):
    df_feat = df.copy()
    
    # Datetime conversions
    df_feat['order_purchase_timestamp'] = pd.to_datetime(df_feat['order_purchase_timestamp'])
    df_feat['order_estimated_delivery_date'] = pd.to_datetime(df_feat['order_estimated_delivery_date'])
    
    # 1. Time Features
    df_feat['purchase_year'] = df_feat['order_purchase_timestamp'].dt.year
    df_feat['purchase_month'] = df_feat['order_purchase_timestamp'].dt.month
    df_feat['purchase_dayofweek'] = df_feat['order_purchase_timestamp'].dt.dayofweek
    df_feat['purchase_hour'] = df_feat['order_purchase_timestamp'].dt.hour
    df_feat['is_weekend'] = df_feat['purchase_dayofweek'].isin([5, 6]).astype(int)
    
    # 2. Estimated Delivery Window (Days)
    df_feat['estimated_delivery_days'] = (
        df_feat['order_estimated_delivery_date'] - df_feat['order_purchase_timestamp']
    ).dt.total_seconds() / (24 * 3600)
    
    # 3. Freight Ratio
    df_feat['freight_ratio'] = df_feat['total_freight'] / (df_feat['total_price'] + 0.001)
    
    return df_feat

train_feat = extract_features(train_df)
val_feat = extract_features(val_df)
test_feat = extract_features(test_df)

print("Engineered features preview:")
print(train_feat[['purchase_month', 'purchase_dayofweek', 'estimated_delivery_days', 'freight_ratio']].head())

Engineered features preview:
   purchase_month  purchase_dayofweek  estimated_delivery_days  freight_ratio
0               9                   3                18.488449       0.062902
1              10                   0                23.593866       0.520384
2              10                   0                34.293866       0.784896
3              10                   0                56.115556       0.472445
4              10                   0                50.079132       0.113093


In [4]:
from sklearn.preprocessing import LabelEncoder

# Fill numerical missing values using TRAIN median
num_impute_cols = ['total_price', 'total_freight', 'num_items', 'total_payment', 'max_installments']
for col in num_impute_cols:
    median_val = train_feat[col].median()
    train_feat[col] = train_feat[col].fillna(median_val)
    val_feat[col] = val_feat[col].fillna(median_val)
    test_feat[col] = test_feat[col].fillna(median_val)

# Categorical Encoding based on Train classes
cat_cols = ['customer_state', 'main_payment_type']
for col in cat_cols:
    train_feat[col] = train_feat[col].fillna('Unknown')
    val_feat[col] = val_feat[col].fillna('Unknown')
    test_feat[col] = test_feat[col].fillna('Unknown')
    
    le = LabelEncoder()
    # Fit strictly on train
    le.fit(train_feat[col].astype(str))
    
    # Transform with fallback for unknown categories
    train_feat[col] = le.transform(train_feat[col].astype(str))
    val_feat[col] = val_feat[col].astype(str).map(lambda s: le.transform([s])[0] if s in le.classes_ else -1)
    test_feat[col] = test_feat[col].astype(str).map(lambda s: le.transform([s])[0] if s in le.classes_ else -1)

print("Encoding and Imputation complete without Data Leakage!")

Encoding and Imputation complete without Data Leakage!


In [5]:
feature_cols = [
    'customer_state', 'total_price', 'total_freight', 'num_items', 
    'num_unique_sellers', 'total_payment', 'max_installments', 'main_payment_type',
    'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour',
    'is_weekend', 'estimated_delivery_days', 'freight_ratio'
]
target_col = 'is_late'

# Prepare final DataFrames
train_final = train_feat[feature_cols + [target_col]]
val_final = val_feat[feature_cols + [target_col]]
test_final = test_feat[feature_cols + [target_col]]

# Save final feature artifacts
train_final.to_csv(os.path.join("artifacts", "train_features.csv"), index=False)
val_final.to_csv(os.path.join("artifacts", "val_features.csv"), index=False)
test_final.to_csv(os.path.join("artifacts", "test_features.csv"), index=False)

print("Final Prepared Artifacts Saved:")
print(f"train_features.csv shape: {train_final.shape}")
print(f"val_features.csv shape:   {val_final.shape}")
print(f"test_features.csv shape:  {test_final.shape}")

Final Prepared Artifacts Saved:
train_features.csv shape: (67534, 16)
val_features.csv shape:   (14472, 16)
test_features.csv shape:  (14472, 16)


In [7]:
import os
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Inputs and target
X_train = train_final[feature_cols]
y_train = train_final[target_col]

# Separate categorical and numerical columns
categorical_cols = [
    "customer_state",
    "main_payment_type"
]

numerical_cols = [
    col for col in feature_cols
    if col not in categorical_cols
]

# Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numerical_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

# Complete model pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

# Train
model.fit(X_train, y_train)

# Save the files required by the API
artifact_dir = "artifacts"
os.makedirs(artifact_dir, exist_ok=True)

joblib.dump(model, os.path.join(artifact_dir, "model.pkl"))
joblib.dump(feature_cols, os.path.join(artifact_dir, "feature_columns.pkl"))

print("Model artifacts saved successfully:")
print(os.path.join(artifact_dir, "model.pkl"))
print(os.path.join(artifact_dir, "feature_columns.pkl"))

Model artifacts saved successfully:
artifacts\model.pkl
artifacts\feature_columns.pkl
